# 06 — Fairness Analysis

This notebook measures **disparate treatment in predictions** across the protected attributes defined in `config.yaml`: sex, race, age, marital status, and native country.

We focus on the **primary model (XGBoost)** saved from `04_modeling.ipynb`, with **logistic regression** as an interpretable baseline. Metrics use [Fairlearn](https://fairlearn.org/) on the held-out test set.

## What we report

- **Selection rate** — fraction predicted positive (`>50K` / class 1); central to demographic parity.
- **Group metrics** — accuracy, precision, recall via `MetricFrame`.
- **Demographic parity difference (DPD)** and **equalized odds difference (EOD)** — global summaries per sensitive attribute.

Numeric **age** is binned with `pd.qcut` on the test set for comparable subgroup sizes (interval labels). Categorical attributes are used as-is.

In [ ]:
import warnings
from functools import partial

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from IPython.display import display

warnings.filterwarnings("ignore")

from sklearn.metrics import accuracy_score, precision_score, recall_score

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate,
)

sns.set_theme(style="whitegrid")
np.random.seed(42)

In [ ]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROTECTED = list(config["protected_attributes"])
TARGET = config["target"]["column"]
MODELS_PATH = config["paths"]["models"].rstrip("/") + "/"
FIGURES_PATH = config["paths"]["figures"]

PRIMARY = "xgboost"
BASELINE = "logistic_regression"

print("Protected attributes:", PROTECTED)
print("Target:", TARGET)

In [ ]:
test = pd.read_csv("../data/processed/test.csv")
X_test = test.drop(columns=[TARGET])
y_raw = test[TARGET]


def as_binary_income(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(int)
    s_str = s.astype(str).str.replace(".", "", regex=False)
    return s_str.str.contains(">50K").astype(int)


y_test = as_binary_income(y_raw)

print(test.shape)
print(y_test.value_counts())

In [ ]:
def load_model(name: str):
    path = f"../{MODELS_PATH}{name}.pkl"
    return joblib.load(path)


models = {}
for key in (PRIMARY, BASELINE):
    try:
        models[key] = load_model(key)
        print(f"Loaded: {key}")
    except Exception as e:
        print(f"Skip {key}: {e}")

if PRIMARY not in models:
    raise RuntimeError(
        f"Expected trained `{PRIMARY}.pkl` under ../{MODELS_PATH}. "
        "Run `04_modeling.ipynb` through the save section (or your training script) first."
    )

In [ ]:
preds = {}
probs = {}

for name, est in models.items():
    raw = est.predict(X_test)
    raw_arr = np.asarray(raw)
    if np.issubdtype(raw_arr.dtype, np.number):
        preds[name] = raw_arr.astype(int)
    else:
        preds[name] = (
            pd.Series(raw).astype(str).str.replace(".", "", regex=False).str.contains(">50K").astype(int).values
        )
    if hasattr(est, "predict_proba"):
        probs[name] = est.predict_proba(X_test)[:, 1]
    else:
        probs[name] = None

y_pred_primary = preds[PRIMARY]
y_pred_baseline = preds.get(BASELINE, y_pred_primary)

In [ ]:
def sensitive_for_attribute(df: pd.DataFrame, attr: str) -> pd.Series:
    """Return a discrete sensitive-features series aligned with df.index."""
    if attr == "age":
        if "age" not in df.columns:
            raise KeyError("Column `age` not found")
        # Quantile bins for comparable subgroup sizes
        q = pd.qcut(df["age"], q=5, duplicates="drop")
        return q.astype(str).rename("age_bin")
    if attr not in df.columns:
        raise KeyError(f"Missing column `{attr}`")
    return df[attr].astype(str)


prec = partial(precision_score, zero_division=0)
rec = partial(recall_score, zero_division=0)

metric_fns = {
    "accuracy": accuracy_score,
    "precision": prec,
    "recall": rec,
    "selection_rate": selection_rate,
}

## 1. Global fairness gaps (DPD & EOD)

Lower absolute values are closer to parity between groups (under the corresponding fairness definition). These are **observational** measures on one test sample — not causal fairness guarantees.

In [ ]:
rows = []
for attr in PROTECTED:
    try:
        s = sensitive_for_attribute(X_test, attr)
    except KeyError as e:
        print(e)
        continue
    for model_name, y_hat in preds.items():
        rows.append(
            {
                "attribute": attr,
                "model": model_name,
                "dpd": demographic_parity_difference(
                    y_true=y_test, y_pred=y_hat, sensitive_features=s
                ),
                "eod": equalized_odds_difference(
                    y_true=y_test, y_pred=y_hat, sensitive_features=s
                ),
            }
        )

summary_global = pd.DataFrame(rows)
if not summary_global.empty:
    summary_global = summary_global.round(4)
    display(summary_global)
    summary_global.to_csv("../reports/fairness_global_dpd_eod.csv", index=False)
    print("Saved ../reports/fairness_global_dpd_eod.csv")

## 2. Group-level metrics (`MetricFrame`)

Heatmaps below: rows are **sensitive groups**, columns are **metrics**, for the primary model.

In [ ]:
for attr in PROTECTED:
    try:
        s = sensitive_for_attribute(X_test, attr)
    except KeyError as e:
        print(e)
        continue

    mf = MetricFrame(
        metrics=metric_fns,
        y_true=y_test,
        y_pred=y_pred_primary,
        sensitive_features=s,
    )
    by_g = mf.by_group.sort_index()

    fig_h = max(4, 0.35 * len(by_g))
    plt.figure(figsize=(10, fig_h))
    sns.heatmap(by_g, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1)
    plt.title(f"Group metrics — {PRIMARY} — sensitive: {attr}")
    plt.ylabel("group")
    plt.tight_layout()
    out = f"../{FIGURES_PATH}fairness_metricframe_{attr}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

    diff = by_g.max() - by_g.min()
    print(f"\nBetween-group range (max - min) for {attr}:\n", diff.round(4))

## 3. Baseline vs primary — selection rate by group

Overlay helps see whether the gradient boosting model **amplifies or reduces** selection-rate disparities relative to logistic regression.

In [ ]:
if BASELINE in preds:
    compare_attr = "sex" if "sex" in PROTECTED else PROTECTED[0]
    s_cmp = sensitive_for_attribute(X_test, compare_attr)

    groups = s_cmp.unique()
    sel_lr = []
    sel_xgb = []
    for g in sorted(groups):
        m = s_cmp == g
        sel_lr.append((preds[BASELINE][m] == 1).mean())
        sel_xgb.append((preds[PRIMARY][m] == 1).mean())

    x = np.arange(len(groups))
    w = 0.35
    plt.figure(figsize=(max(8, len(groups)), 5))
    plt.bar(x - w / 2, sel_lr, width=w, label=BASELINE)
    plt.bar(x + w / 2, sel_xgb, width=w, label=PRIMARY)
    plt.xticks(x, sorted(groups), rotation=35, ha="right")
    plt.ylabel("selection rate (P(predicted positive))")
    plt.title(f"Selection rate by {compare_attr}")
    plt.legend()
    plt.tight_layout()
    out = f"../{FIGURES_PATH}fairness_selection_rate_{compare_attr}_lr_vs_xgb.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
else:
    print("Baseline model not loaded; skip comparison chart.")

## 4. Takeaways (fill in after you run)

- Which attribute shows the **largest** `dpd` / `eod` for XGBoost?
- Does the baseline **agree** on direction of disparity, or does non-linearity change the pattern?
- **Next step (per README):** explainability — global/local SHAP, counterfactuals, and error analysis by subgroup — in a follow-on notebook or script when you add it.

---

**Outputs:** `../reports/fairness_global_dpd_eod.csv`, figures under `../reports/figures/fairness_*`.